# Backup coverage: how many independent options are nearby?

This lab uses synthetic travel costs. It contrasts ordinary coverage (at least one selected facility) with two-cover redundancy (at least two distinct selected facilities). The model counts potential alternatives; it does not estimate outage probabilities, capacity or queueing.

Rows are demand locations; columns are candidate facilities. All costs and the threshold use the same arbitrary travel-time unit.

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd().parent / 'implementations' / 'python' / 'backup-coverage'))
from backup_coverage import solve_backup_coverage
import pandas as pd


In [ ]:
costs = [
    [2, 3, 9, 12],
    [1, 8, 2, 9],
    [8, 1, 3, 10],
    [7, 9, 2, 3],
    [9, 8, 7, 1],
]
weights = [10, 5, 8, 4, 6]
threshold = 4
budget = 3
print('Demand rows:', len(costs), '| Candidate facilities:', len(costs[0]))
print('Demand weights:', weights)


## Compare one-cover and two-cover

For each redundancy requirement, inspect selected candidates, covered weight and the number of selected facilities within the threshold for every demand row.

In [ ]:
rows = []
for r in (1, 2, 3):
    sites, value, covered, counts = solve_backup_coverage(
        costs, p=budget, service_threshold=threshold,
        required_cover_count=r, demand_weights=weights
    )
    rows.append({'required facilities': r, 'selected candidates': sites,
                 'covered weight': value, 'covered rows': covered,
                 'eligible facility counts': counts})
pd.DataFrame(rows)


## Sensitivity to the service threshold

The threshold is a service-policy assumption, not a value to tune until the map looks attractive. Compare several plausible thresholds and record the justification for each.

In [ ]:
sensitivity = []
for limit in (2, 4, 6):
    sites, value, covered, counts = solve_backup_coverage(
        costs, p=budget, service_threshold=limit,
        required_cover_count=2, demand_weights=weights
    )
    sensitivity.append({'threshold': limit, 'sites': sites,
                        'backup-covered weight': value,
                        'covered demand rows': sum(covered),
                        'counts': counts})
pd.DataFrame(sensitivity)


## Interpret the result

1. Which demand rows are reached by one selected facility but not two?
2. Which candidate sites are responsible for the second route to a demand row?
3. Could two selected facilities fail together because they share a bridge, power supply, hazard zone or staff pool?
4. What information would be needed to estimate the probability of maintaining service after an outage?

**Important:** r-covering is a deterministic redundancy screen. A count of two eligible facilities is not a 100% guarantee, nor is it an outage-survival probability. A reliability model needs failure scenarios or probabilities, dependence assumptions and service-capacity information.